# Gambit Vision: overview

Overhead camera image -> legal board state (FEN), for the Gambit chess robot. This folder
holds the whole vision package as notebooks (conventions: `docs/notebooks.md` at the repo
root). **Library notebooks** hold definitions plus a fast `## Self-check`; **run_\*
notebooks** use the camera, a GPU or recorded data and are not executed in CI.

Docs: [`../docs/contracts.md`](../docs/contracts.md) (data formats between stages),
[`../docs/M3_TODO.md`](../docs/M3_TODO.md), [`../docs/M4_TODO.md`](../docs/M4_TODO.md),
[`../docs/M5_TODO.md`](../docs/M5_TODO.md). Configs: `../configs/camera.yaml`,
`../configs/train.yaml`; board geometry: `gambit_bringup/config/board.yaml`.

## Deliverable checklist

Every notebook below is a design skeleton: constants, dataclasses, signatures and design
notes are in place; every function body is a TODO (`raise NotImplementedError`) and every
`## Self-check` is a test plan for you to turn into asserts. Only `config.ipynb` (YAML
plumbing) is implemented.

**M1: calibration + board grid**
- [ ] Print markers and ChArUco board: `markers.ipynb`
- [ ] Camera intrinsics + reprojection error: `calibration.ipynb` -> `run_calibrate.ipynb`
- [ ] ArUco corners -> homography -> 8x8 grid overlay + square-center pixel error:
      `homography.ipynb` (synthetic board for testing, 3-marker fallback) ->
      `run_grid_overlay.ipynb`

**M2: data collection**
- [ ] Crops per `docs/contracts.md`, auto-labels from PGN replay, manifest writer, live loop:
      `capture.ipynb` -> `run_capture.ipynb`
- [ ] Sim sessions from Gazebo (`gambit_sim` data generator) in the same format

**M3: square classifiers**
- [ ] Session-split dataset, transforms, external loaders: `dataset.ipynb`
- [ ] Color/edge baseline (features + sklearn classifier): `baseline.ipynb` ->
      `run_train_baseline.ipynb`
- [ ] ResNet-18, 13 classes, three experiments sim/real/mixed: `train.ipynb` ->
      `run_train_cnn.ipynb` (Colab)

**M4: board assembly + evaluation**
- [ ] 64 predictions -> FEN, legality, low-confidence flags, repair: `board_assembly.ipynb`
- [ ] Metrics, confusion matrix, results folder: `evaluate.ipynb` -> `run_evaluate.ipynb`
- [ ] chesscog comparison: `chesscog_compare.ipynb` -> `run_chesscog.ipynb`

**M5 (optional): move detection**
- [ ] Stable frames + legal-move inference + tracker + game evaluation:
      `move_detection.ipynb` -> `run_move_detection.ipynb`

**Full design: perception and rules (README)**
- [ ] Identify the board state from the camera: M1 -> M4 pipeline above
- [ ] Verify physical placement (piece centered within tolerance): `placement.ipynb`
      (tolerance math + detection)
- [ ] Verify rule legality, separate from placement: `board_assembly.ipynb: check_legality`
      (placement plausibility) and `move_detection.ipynb: infer_move` (legal moves only)
- [ ] Detect camera vs engine mismatches and recover: `board_sync.ipynb` (diff,
      classification, recovery policy)
- [ ] Optional alternative detector: `yolo.ipynb` + `configs/yolo_data.yaml` (repo root)

## Run order

| step | notebook | needs | produces |
|---|---|---|---|
| 1 | `markers.ipynb` (set `WRITE = True`) | - | marker PNGs, `charuco_board.png` to print |
| 2 | `run_calibrate.ipynb` | camera, printed ChArUco | `calibration/camera_intrinsics.yaml` |
| 3 | `run_grid_overlay.ipynb` | camera, board with markers | overlay screenshot, square-center error |
| 4 | `run_capture.ipynb` | camera, `data/games/*.pgn` | `data/crops/<session>/` |
| 5 | `run_train_baseline.ipynb` | crops, session lists in `train.yaml` | `models/baseline/baseline.pkl` |
| 6 | `run_train_cnn.ipynb` (Colab GPU) | crops | `models/resnet18/<run>/best.pt` |
| 7 | `run_evaluate.ipynb` | models, test sessions | `results/<run>/` |
| 8 | `run_chesscog.ipynb` (separate env) | test frames, our `boards.csv` | `chesscog_vs_ours.csv` |
| 9 | `run_move_detection.ipynb` (optional) | recorded games, model | `move_metrics.json` |

Library dependencies (`%run`): `labels` <- `dataset` <- `train`, `evaluate`, `capture`;
`config` <- `homography` <- `capture`, `placement`, `markers`; `calibration` <- `markers`.

## Data flow

```
camera frame --undistort (calibration)--> image
  image --ArUco markers--> H (homography) --warp--> top-down board (warp)
  warp --crop_box x 64 (capture)--> square crops --[PGN replay labels]--> data/crops/<session>/manifest.csv
  crops --(dataset: session splits)--> baseline / ResNet-18 (train) --> (64, 13) probabilities
  probabilities --assemble (board_assembly)--> BoardPrediction (FEN, confidence, legal)
       |--> evaluate / chesscog_compare --> results/<run>/
       |--> infer_move (move_detection) --> move for the engine
       |--> board_sync (vs engine board) --> ok / accept move / ask human / auto-correct
  warp --piece base point (placement)--> centered within tolerance?
```

## Self-check: every notebook named above exists

In [ ]:
from pathlib import Path

_ov_names = ["config", "labels", "calibration", "homography", "markers", "capture", "dataset",
             "baseline", "train", "board_assembly", "evaluate", "chesscog_compare",
             "move_detection", "placement", "board_sync", "yolo",
             "run_calibrate", "run_grid_overlay", "run_capture", "run_train_baseline",
             "run_train_cnn", "run_evaluate", "run_chesscog", "run_move_detection"]
_ov_missing = [n for n in _ov_names if not Path(f"{n}.ipynb").exists()]
assert not _ov_missing, _ov_missing
assert all(Path(p).exists() for p in ["../docs/contracts.md", "../docs/M3_TODO.md",
                                      "../docs/M4_TODO.md", "../docs/M5_TODO.md",
                                      "../../configs/yolo_data.yaml"])
print("overview OK:", len(_ov_names), "notebooks")